# 02 - LLM Knowledge Extraction

**Task 1, Fase 2:** Mengekstrak entitas dan relasi dari chunks UU ITE menggunakan Gemini API, lalu memvalidasi dan mendeduplikasi hasilnya.

**Pipeline:** `Chunks → LLM Extract → Validate → Deduplicate`

In [1]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
print(f'Project root: {PROJECT_ROOT}')
print(f'API Key: {GEMINI_API_KEY[:10]}...' if GEMINI_API_KEY else 'ERROR: No API key found!')

Project root: d:\TA\llm-driven-legal-kg-visualization
API Key: AIzaSyB38G...


## Step 1: Load Chunks

In [2]:
# Load chunks dari Fase 1
CHUNKS_PATH = 'data/chunks/UU_11_2008_chunks.json'

with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

chunks = chunks_data['chunks']
print(f'Document: {chunks_data["document_id"]}')
print(f'Total chunks: {len(chunks)}')
print(f'Token stats: {chunks_data.get("stats", {})}')

Document: UU_11_2008
Total chunks: 28
Token stats: {'min_tokens': 436, 'max_tokens': 2272, 'avg_tokens': 795.5}


## Step 2: Extract KG Triples with Gemini

Mengirim chunks ke Gemini API dengan system prompt ontologi. Output: nodes (entities) dan edges (relations).

⚠️ **Proses ini membutuhkan API calls dan memakan waktu beberapa menit.**

In [3]:
from pipeline.transform.llm_extractor import extract_all_triples

# Extract triples dari semua chunks
triples_path = extract_all_triples(
    chunks_path=CHUNKS_PATH,
    output_dir='data/triples',
    api_key=GEMINI_API_KEY,
    model_name='gemini-2.5-flash',
    batch_size=3,         # 3 chunks per API call
    max_retries=3,
    delay_between_calls=2.0  # 2 detik antar call (rate limiting)
)

# Load results
with open(triples_path, 'r', encoding='utf-8') as f:
    triples_data = json.load(f)

print(f'\n=== Extraction Summary ===')
print(f'Total Nodes: {triples_data["total_nodes"]}')
print(f'Total Edges: {triples_data["total_edges"]}')
print(f'Errors: {len(triples_data.get("errors", []))}')
print(f'Saved to: {triples_path}')

c:\Users\daffarafi\miniconda3\envs\ta-skripsi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\TA\llm-driven-legal-kg-visualization\pipeline\transform\llm_extractor.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
Extracting UU_11_2008: 100%|██████████| 10/10 [14:39<00:00, 87.94s/it]


=== Extraction Summary ===
Total Nodes: 624
Total Edges: 759
Errors: 0
Saved to: data/triples\UU_11_2008_triples.json


In [4]:
# Preview nodes by type
from collections import Counter

node_types = Counter(n['type'] for n in triples_data['nodes'])
edge_types = Counter(e['type'] for e in triples_data['edges'])

print('=== Node Types ===')
for t, count in node_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\n=== Edge Types ===')
for t, count in edge_types.most_common():
    print(f'  {t:20s}: {count}')

# Sample nodes
print(f'\n=== Sample Nodes (first 10) ===')
for n in triples_data['nodes'][:10]:
    print(f'  [{n["type"]}] {n["label"]}')

=== Node Types ===
  Ayat                : 171
  PerbuatanHukum      : 150
  Pasal               : 120
  EntitasHukum        : 83
  KonsepHukum         : 60
  Sanksi              : 16
  Bab                 : 14
  UndangUndang        : 10

=== Edge Types ===
  MEMILIKI_AYAT       : 170
  BERLAKU_UNTUK       : 159
  MENGATUR            : 155
  MERUJUK             : 134
  MEMUAT              : 71
  MENDEFINISIKAN      : 51
  MENETAPKAN_SANKSI   : 19

=== Sample Nodes (first 10) ===
  [UndangUndang] Undang-Undang tentang Informasi dan Transaksi Elektronik
  [Bab] BAB I KETENTUAN UMUM
  [Pasal] Pasal 1
  [KonsepHukum] Informasi Elektronik
  [KonsepHukum] Transaksi Elektronik
  [KonsepHukum] Teknologi Informasi
  [KonsepHukum] Dokumen Elektronik
  [KonsepHukum] Sistem Elektronik
  [KonsepHukum] Penyelenggaraan Sistem Elektronik
  [KonsepHukum] Jaringan Sistem Elektronik


## Step 3: Validate Triples

In [5]:
from pipeline.transform.validator import validate_triples_file

validated_path = validate_triples_file(
    input_path=triples_path,
    output_dir='data/validated',
    strict=False  # Set True for stricter edge type checking
)

with open(validated_path, 'r', encoding='utf-8') as f:
    validated_data = json.load(f)

print(f'=== Validation Summary ===')
print(f'Nodes: {validated_data["total_nodes"]} (removed {validated_data["removed_nodes"]})')
print(f'Edges: {validated_data["total_edges"]} (removed {validated_data["removed_edges"]})')
print(f'Saved to: {validated_path}')

# Show errors if any
error_log = validated_path.replace('_triples.json', '_errors.log')
if os.path.exists(error_log):
    with open(error_log, 'r', encoding='utf-8') as f:
        errors = f.read()
    print(f'\n=== Errors ({errors.count(chr(10))} lines) ===')
    print(errors[:1000])

=== Validation Summary ===
Nodes: 624 (removed 0)
Edges: 589 (removed 170)
Saved to: data/validated\UU_11_2008_triples.json

=== Errors (174 lines) ===
Validation Errors for UU_11_2008
Total errors: 170

1. Invalid edge type 'MEMILIKI_AYAT': Pasal_5 -> Ayat_5_1
2. Invalid edge type 'MEMILIKI_AYAT': Pasal_5 -> Ayat_5_2
3. Invalid edge type 'MEMILIKI_AYAT': Pasal_5 -> Ayat_5_3
4. Invalid edge type 'MEMILIKI_AYAT': Pasal_5 -> Ayat_5_4
5. Invalid edge type 'MEMILIKI_AYAT': Pasal_8 -> Ayat_8_1
6. Invalid edge type 'MEMILIKI_AYAT': Pasal_8 -> Ayat_8_2
7. Invalid edge type 'MEMILIKI_AYAT': Pasal_8 -> Ayat_8_3
8. Invalid edge type 'MEMILIKI_AYAT': Pasal_8 -> Ayat_8_4
9. Invalid edge type 'MEMILIKI_AYAT': Pasal_10 -> Ayat_10_1
10. Invalid edge type 'MEMILIKI_AYAT': Pasal_10 -> Ayat_10_2
11. Invalid edge type 'MEMILIKI_AYAT': Pasal_10 -> Ayat_10_1
12. Invalid edge type 'MEMILIKI_AYAT': Pasal_10 -> Ayat_10_2
13. Invalid edge type 'MEMILIKI_AYAT': Pasal_11 -> Ayat_11_1
14. Invalid edge type 'MEMIL

## Step 4: Deduplicate Entities

In [6]:
from pipeline.transform.deduplicator import deduplicate_triples_file

deduped_path = deduplicate_triples_file(
    input_path=validated_path,
    output_dir='data/deduped',
    similarity_threshold=0.85
)

with open(deduped_path, 'r', encoding='utf-8') as f:
    deduped_data = json.load(f)

print(f'=== Deduplication Summary ===')
print(f'Nodes: {deduped_data["nodes_before"]} → {deduped_data["total_nodes"]} (merged {deduped_data["nodes_merged"]})')
print(f'Edges: {deduped_data["edges_before"]} → {deduped_data["total_edges"]}')
print(f'Saved to: {deduped_path}')

=== Deduplication Summary ===
Nodes: 624 → 462 (merged 8)
Edges: 589 → 571
Saved to: data/deduped\UU_11_2008_triples.json


In [7]:
# Final KG overview
print('=== Final Knowledge Graph ===')
final_node_types = Counter(n['type'] for n in deduped_data['nodes'])
final_edge_types = Counter(e['type'] for e in deduped_data['edges'])

print(f'\nNode Types ({deduped_data["total_nodes"]} total):')
for t, count in final_node_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\nEdge Types ({deduped_data["total_edges"]} total):')
for t, count in final_edge_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\n=== Sample Triples ===')
node_map = {n['id']: n['label'] for n in deduped_data['nodes']}
for e in deduped_data['edges'][:15]:
    src = node_map.get(e.get('source_id', e.get('source', '')), e.get('source_id', '?'))
    tgt = node_map.get(e.get('target_id', e.get('target', '')), e.get('target_id', '?'))
    print(f'  {src} —[{e["type"]}]→ {tgt}')

=== Final Knowledge Graph ===

Node Types (462 total):
  PerbuatanHukum      : 148
  Ayat                : 108
  EntitasHukum        : 67
  Pasal               : 54
  KonsepHukum         : 53
  Sanksi              : 16
  Bab                 : 13
  UndangUndang        : 3

Edge Types (571 total):
  MENGATUR            : 153
  BERLAKU_UNTUK       : 152
  MERUJUK             : 130
  MEMUAT              : 70
  MENDEFINISIKAN      : 48
  MENETAPKAN_SANKSI   : 18

=== Sample Triples ===
  Undang-Undang tentang Informasi dan Transaksi Elektronik —[MEMUAT]→ BAB I KETENTUAN UMUM
  BAB I KETENTUAN UMUM —[MEMUAT]→ Pasal 1
  Pasal 1 —[MENDEFINISIKAN]→ Informasi Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Transaksi Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Teknologi Informasi
  Pasal 1 —[MENDEFINISIKAN]→ Dokumen Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Sistem Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Penyelenggaraan Sistem Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Jaringan Sistem Elektronik
  Pasal 1 —[MENDEF

## Summary

Fase 2 selesai! Output tersimpan di:
- `data/triples/UU_11_2008_triples.json` — Raw LLM extraction
- `data/validated/UU_11_2008_triples.json` — Validated (invalid types removed)
- `data/deduped/UU_11_2008_triples.json` — Deduplicated (final)

**Next:** Run `03_neo4j_ingestion.ipynb` to embed and load into Neo4j.